# Data Prep for Neural Network

In [1]:
import polars as pl

## Train Test Split

Here we perform the split into train, validation and test data, with a **split** of 70% train, 15% validation and 15% test data. The split is performed **random**. 

In [2]:
DATASET = "../data/processed_data/GOLD_HOURLY_DEMAND_COMMUNITY_AREA.parquet"
OUTPUT = "../data/train_test_data/"
TARGET_COL = "trip_count"

SEED = 42
RANDOM = False

In [3]:
if(RANDOM == True):
    # split randomly
    df_split = (
        pl.scan_parquet(DATASET)
        .with_row_index("_row_id")
        .with_columns(
            (pl.col("_row_id").hash(seed=SEED) % 100).alias("_split_bucket")
        )
    )

    train = (
        df_split
        .filter(pl.col("_split_bucket") < 70)
        .drop(["_row_id", "_split_bucket"])
    )

    val = (
        df_split
        .filter(
            (pl.col("_split_bucket") >= 70) &
            (pl.col("_split_bucket") < 85)
        )
        .drop(["_row_id", "_split_bucket"])
    )

    test = (
        df_split
        .filter(pl.col("_split_bucket") >= 85)
        .drop(["_row_id", "_split_bucket"])
    )
else :
    # split according to time
    df_split = pl.scan_parquet(DATASET)

    train = df_split.filter(
        pl.col("datetime_hour") < pl.datetime(2025, 9, 1)
    )

    val = df_split.filter(
        (pl.col("datetime_hour") >= pl.datetime(2025, 9, 1)) &
        (pl.col("datetime_hour") < pl.datetime(2026, 1, 1))
    )

    test = df_split.filter(
        pl.col("datetime_hour") >= pl.datetime(2026, 1, 1)
    )


total_count = df_split.select(pl.len()).collect().item()
train_count = train.select(pl.len()).collect().item()
val_count = val.select(pl.len()).collect().item()
test_count = test.select(pl.len()).collect().item()

print("Total:", total_count)
print("Train:", train_count, " Share: ", round(train_count / total_count,2))
print("Val:", val_count, " Share: ", round(val_count / total_count,2))
print("Test:", test_count, " Share: ", round(test_count / total_count, 2))

train.sink_parquet(OUTPUT + "train.parquet")
val.sink_parquet(OUTPUT + "val.parquet")
test.sink_parquet(OUTPUT + "test.parquet")

Total: 1574265
Train: 1125278  Share:  0.71
Val: 225456  Share:  0.14
Test: 223531  Share:  0.14


In [4]:
type(train)

polars.lazyframe.frame.LazyFrame

In [5]:
df_split.head(10).collect()

datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,vsby,p01m,skyc1_BKN,skyc1_CLR,skyc1_FEW,skyc1_OVC,skyc1_SCT,skyc1_VV,date,is_holiday,community_area,food_drink,landmark,shop,train_station,trip_count,trip_seconds_sum,trip_seconds_mean,trip_seconds_min,trip_seconds_max,trip_miles_sum,trip_miles_mean,trip_miles_min,trip_miles_max,fare_sum,fare_mean,fare_min,fare_max,tips_sum,tips_mean,tips_min,tips_max,tolls_sum,tolls_mean,tolls_min,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
datetime[μs],i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,date,i8,i64,f64,f64,f64,f64,u32,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
2024-08-02 23:00:00,8,5,23,-0.5,-0.866025,-0.433884,-0.900969,-0.258819,0.965926,24.44,66.38,3.0,10.0,0.0,0,1,0,0,0,0,2024-08-02,0,66,18.0,5.0,20.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2024-08-03 14:00:00,8,6,14,-0.5,-0.866025,-0.974928,-0.222521,-0.5,-0.866025,31.11,51.65,8.0,10.0,0.0,0,0,0,0,1,0,2024-08-03,0,66,18.0,5.0,20.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2024-09-11 05:00:00,9,3,5,-0.866025,-0.5,0.974928,-0.222521,0.965926,0.258819,16.67,62.36,0.0,10.0,0.0,0,0,1,0,0,0,2024-09-11,0,66,18.0,5.0,20.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2024-08-03 11:00:00,8,6,11,-0.5,-0.866025,-0.974928,-0.222521,0.258819,-0.965926,31.11,49.9,3.0,10.0,0.0,0,0,1,0,0,0,2024-08-03,0,66,18.0,5.0,20.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2024-08-03 22:00:00,8,6,22,-0.5,-0.866025,-0.974928,-0.222521,-0.5,0.866025,25.56,62.12,6.0,10.0,0.0,0,0,1,0,0,0,2024-08-03,0,66,18.0,5.0,20.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2024-08-05 17:00:00,8,1,17,-0.5,-0.866025,0.0,1.0,-0.965926,-0.258819,26.835,66.19,12.5,10.0,0.0,0,0,1,0,0,0,2024-08-05,0,66,18.0,5.0,20.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2024-08-03 02:00:00,8,6,2,-0.5,-0.866025,-0.974928,-0.222521,0.5,0.866025,23.33,78.78,0.0,10.0,0.0,0,1,0,0,0,0,2024-08-03,0,66,18.0,5.0,20.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2024-08-03 08:00:00,8,6,8,-0.5,-0.866025,-0.974928,-0.222521,0.866025,-0.5,27.78,62.62,4.0,10.0,0.0,0,1,0,0,0,0,2024-08-03,0,66,18.0,5.0,20.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2024-09-19 20:00:00,9,4,20,-0.866025,-0.5,0.433884,-0.900969,-0.866025,0.5,25.56,50.22,7.0,10.0,0.0,0,0,0,0,1,0,2024-09-19,0,66,18.0,5.0,20.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""


## Feature Selection